# What is concurrency?

# Experiment 1: Creating and observing threads

A normal Python program already has one thread: the main thread.
    
    Python Process
    │
    ├── Main Thread
    │
    │   execute statement 1
    │   execute statement 2
    │   execute statement 3
    │
    └── Shared Process Memory
        ├── objects
        ├── globals
        ├── heap
        └── imported modules

    All threads share:
        process memory
        Python objects
        globals
        heap
    
    Each thread has its own:
        call stack
        current execution position

In [1]:
import threading
import time
from datetime import datetime


def log(message):
    """
    Small helper so that every message shows:
    - current time
    - current thread name
    - what that thread is doing
    """

    current_thread = threading.current_thread()
    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(
        f"{timestamp} | "
        f"{current_thread.name:<12} | "
        f"{message}"
    )


def worker(task_name, duration):
    """
    Function that will be executed by a worker thread.
    """

    log(f"START {task_name}")

    # sleep simulates some slow operation such as:
    # - network request
    # - database query
    # - file read
    # - waiting for an external service
    time.sleep(duration)

    log(f"END   {task_name}")


# ---------------------------------------------------------
# STEP 1: Observe the main thread
# ---------------------------------------------------------

log("Program started")


# ---------------------------------------------------------
# STEP 2: Create thread objects
#
# Important:
# Creating a Thread object does NOT start the thread.
# ---------------------------------------------------------

thread1 = threading.Thread(
    target=worker,
    args=("Task-A", 3),
    name="Worker-1"
)

thread2 = threading.Thread(
    target=worker,
    args=("Task-B", 2),
    name="Worker-2"
)


log("Thread objects created")


# ---------------------------------------------------------
# STEP 3: Start the threads
#
# start() tells Python:
# "This thread is now eligible to execute."
# ---------------------------------------------------------

thread1.start()

log("Worker-1 started")

thread2.start()

log("Worker-2 started")


# ---------------------------------------------------------
# STEP 4: Main thread continues executing
#
# start() does NOT wait for the worker to finish.
# ---------------------------------------------------------

log("Main thread continues doing its own work")

time.sleep(1)

log("Main thread finished its small task")


# ---------------------------------------------------------
# STEP 5: Wait for worker threads
# join() means:
# "The current thread should wait until this other
# thread finishes."
# Here the current thread is MainThread.
# ---------------------------------------------------------

log("Main thread waiting for Worker-1")

thread1.join()

log("Worker-1 has finished")


log("Main thread waiting for Worker-2")

thread2.join()

log("Worker-2 has finished")


log("Program finished")

06:11:07.779 | MainThread   | Program started
06:11:07.784 | MainThread   | Thread objects created
06:11:07.785 | Worker-1     | START Task-A
06:11:07.786 | MainThread   | Worker-1 started
06:11:07.786 | Worker-2     | START Task-B
06:11:07.786 | MainThread   | Worker-2 started
06:11:07.786 | MainThread   | Main thread continues doing its own work
06:11:08.790 | MainThread   | Main thread finished its small task
06:11:08.790 | MainThread   | Main thread waiting for Worker-1
06:11:09.791 | Worker-2     | END   Task-B
06:11:10.792 | Worker-1     | END   Task-A
06:11:10.793 | MainThread   | Worker-1 has finished
06:11:10.793 | MainThread   | Main thread waiting for Worker-2
06:11:10.794 | MainThread   | Worker-2 has finished
06:11:10.794 | MainThread   | Program finished


In [3]:
str(datetime.now())

'2026-09-02 06:17:21.946877'

In [34]:
def log(message):
    curr_time = str(datetime.now())
    current_thread = threading.current_thread()
    active_threads = threading.active_count()
    print(f"{curr_time} | {message} | thread = {current_thread.name} | active threads: {active_threads} ")


def some_task(task_name, duration):
    log(f"Starting task: {task_name}")
    time.sleep(duration)

    log(f"finished task: {task_name}")

In [29]:
## sequential operation

log("Starting the breakfast process")

some_task("making coffee", 5)
some_task("making toast", 10)


log("finished the breakfast process")


2026-09-02 06:40:44.802125 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 06:40:44.802269 | Starting task: making coffee | thread = MainThread | active threads: 9 
2026-09-02 06:40:49.806632 | finished task: making coffee | thread = MainThread | active threads: 9 
2026-09-02 06:40:49.807320 | Starting task: making toast | thread = MainThread | active threads: 9 
2026-09-02 06:40:59.808496 | finished task: making toast | thread = MainThread | active threads: 9 
2026-09-02 06:40:59.809165 | finished the breakfast process | thread = MainThread | active threads: 9 


In [35]:
## threaded operation

from threading import Thread


log("Starting the breakfast process")

## create a thread

log("Creating the thread objects")

coffee_thread = threading.Thread(target = some_task, args =  ("making coffee", 5), name = "coffee_thread") 
toast_thread = threading.Thread(target = some_task, args =  ("making toast", 10), name = "toast_thread") 

log("Created the thread objects")


log("Starting the thread objects!!")

coffee_thread.start()
log("coffee thread started!!")

toast_thread.start()
log("toast thread started!!")


## try adding thread.join

coffee_thread.join()
toast_thread.join()

log("finished the breakfast process!!!")

2026-09-02 07:04:08.438949 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439077 | Creating the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439295 | Created the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439362 | Starting the thread objects!! | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439588 | Starting task: making coffee | thread = coffee_thread | active threads: 10 
2026-09-02 07:04:08.439846 | coffee thread started!! | thread = MainThread | active threads: 10 
2026-09-02 07:04:08.439987 | Starting task: making toast | thread = toast_thread | active threads: 11 
2026-09-02 07:04:08.440137 | toast thread started!! | thread = MainThread | active threads: 11 
2026-09-02 07:04:13.444007 | finished task: making coffee | thread = coffee_thread | active threads: 11 
2026-09-02 07:04:18.443889 | finished task: making toast | thread = toast_thread | active 

In [36]:
print(dir(threading))

['Barrier', 'BoundedSemaphore', 'BrokenBarrierError', 'Condition', 'Event', 'ExceptHookArgs', 'Lock', 'RLock', 'Semaphore', 'TIMEOUT_MAX', 'Thread', 'ThreadError', 'Timer', 'WeakSet', '_CRLock', '_DeleteDummyThreadOnDel', '_DummyThread', '_HAVE_THREAD_NATIVE_ID', '_LockType', '_MainThread', '_PyRLock', '_RLock', '_SHUTTING_DOWN', '_ThreadHandle', '__all__', '__builtins__', '__cached__', '__doc__', '__excepthook__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_active', '_active_limbo_lock', '_after_fork', '_allocate_lock', '_count', '_counter', '_daemon_threads_allowed', '_dangling', '_deque', '_enumerate', '_get_main_thread_ident', '_is_main_interpreter', '_limbo', '_main_thread', '_make_invoke_excepthook', '_make_thread_handle', '_newname', '_os', '_profile_hook', '_register_atexit', '_shutdown', '_start_joinable_thread', '_sys', '_thread_local_info', '_thread_shutdown', '_threading_atexits', '_time', '_trace_hook', 'activeCount', 'active_count', 'currentThread',

In [17]:
threading.active_count()

9

#### see the threads a little better

In [31]:
import threading
import time
import os
from datetime import datetime


def show_threads(label):
    """
    Display what Python currently knows about its threads.
    """

    print(f"\n{'=' * 70}")
    print(label)
    print(f"{'=' * 70}")

    print(f"Process ID (PID): {os.getpid()}")
    print(f"Active thread count: {threading.active_count()}")

    for thread in threading.enumerate():
        print(
            f"  Thread name={thread.name!r}, "
            f"ident={thread.ident}, "
            f"native_id={thread.native_id}, "
            f"alive={thread.is_alive()}"
        )


def some_task():
    print(
        f"\nWorker executing: "
        f"name={threading.current_thread().name}, "
        f"ident={threading.current_thread().ident}, "
        f"native_id={threading.current_thread().native_id}"
    )

    time.sleep(50)

# ---------------------------------------------------------
# STEP 1 - Inspect the process before creating our Thread object.
# ---------------------------------------------------------

show_threads("1. BEFORE creating Thread object")

# ---------------------------------------------------------
# STEP 2 - Create a Thread object.
# This does NOT start an OS thread yet.
# ---------------------------------------------------------

worker = threading.Thread(
    target=some_task,
    name="CoffeeThread"
)

show_threads("2. AFTER creating Thread object")
print(
    "\nPython Thread object says:"
    f" name={worker.name!r},"
    f" ident={worker.ident},"
    f" native_id={worker.native_id},"
    f" alive={worker.is_alive()}"
)


# ---------------------------------------------------------
# STEP 3 - Actually start the thread.
# ---------------------------------------------------------

print("\nCalling worker.start()...")

worker.start()


show_threads("3. AFTER worker.start()")


# ---------------------------------------------------------
# STEP 4 - Wait for the worker to finish.
# ---------------------------------------------------------

worker.join()


show_threads("4. AFTER worker.join()")


1. BEFORE creating Thread object
Process ID (PID): 50970
Active thread count: 9
  Thread name='MainThread', ident=8269492800, native_id=580585, alive=True
  Thread name='IOPub', ident=6140538880, native_id=580693, alive=True
  Thread name='Heartbeat', ident=6157365248, native_id=580694, alive=True
  Thread name='Thread-2 (_watch_pipe_fd)', ident=6175338496, native_id=580697, alive=True
  Thread name='Thread-3 (_watch_pipe_fd)', ident=6192164864, native_id=580698, alive=True
  Thread name='Control', ident=6208991232, native_id=580699, alive=True
  Thread name='Shell channel', ident=6225817600, native_id=580700, alive=True
  Thread name='IPythonHistorySavingThread', ident=6242643968, native_id=580721, alive=True
  Thread name='Thread-1', ident=6260043776, native_id=580725, alive=True

2. AFTER creating Thread object
Process ID (PID): 50970
Active thread count: 9
  Thread name='MainThread', ident=8269492800, native_id=580585, alive=True
  Thread name='IOPub', ident=6140538880, native_id=

In [ ]:
##### ident versus native_id


     - On macOS/Linux/Windows, native_id corresponds to the thread identifier assigned by the underlying operating system.
     - start() is the point at which the thread actually comes into existence as an executing thread.

#### seeing the threads in terminal 
get the pid - `os.getpid()`

ps -M <PID>

#### observing the thread lifecycle

In [32]:
import threading
import time
import os
from datetime import datetime


def log(message):
    """
    Print useful information about the current execution context.
    """

    now = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    current = threading.current_thread()
    print(
        f"{now} | "
        f"PID={os.getpid()} | "
        f"Thread={current.name} | "
        f"native_id={current.native_id} | "
        f"{message}"
    )


def worker():
    """
    Worker function executed by our thread.
    """

    log("Worker function STARTED")

    # Keep the thread alive for 10 seconds.
    # During this period the thread is alive, but sleeping.
    time.sleep(10)

    log("Worker function FINISHED")


# ==========================================================
# STEP 1: Create the Thread object
# ==========================================================

worker_thread = threading.Thread(
    target=worker,
    name="Worker-1"
)

print("\n--- AFTER CREATING THREAD OBJECT ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


# ==========================================================
# STEP 2: Start the thread
# ==========================================================

print("\n--- CALLING start() ---")

worker_thread.start()

print("\n--- IMMEDIATELY AFTER start() ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


# ==========================================================
# STEP 3: MainThread continues
# ==========================================================

print("\n--- MAIN THREAD CONTINUES ---")

for i in range(3):

    print(
        f"MainThread doing work {i + 1} | "
        f"Worker alive={worker_thread.is_alive()}"
    )

    time.sleep(1)


# ==========================================================
# STEP 4: Wait for the worker
# ==========================================================

print("\n--- MAIN THREAD CALLING join() ---")

worker_thread.join()

print("\n--- AFTER join() ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


--- AFTER CREATING THREAD OBJECT ---
name      : Worker-1
ident     : None
native_id : None
alive     : False

--- CALLING start() ---
07:01:27.156 | PID=50970 | Thread=Worker-1 | native_id=769079 | Worker function STARTED

--- IMMEDIATELY AFTER start() ---
name      : Worker-1
ident     : 6344175616
native_id : 769079
alive     : True

--- MAIN THREAD CONTINUES ---
MainThread doing work 1 | Worker alive=True
MainThread doing work 2 | Worker alive=True
MainThread doing work 3 | Worker alive=True

--- MAIN THREAD CALLING join() ---
07:01:37.160 | PID=50970 | Thread=Worker-1 | native_id=769079 | Worker function FINISHED

--- AFTER join() ---
name      : Worker-1
ident     : 6344175616
native_id : 769079
alive     : False


In [40]:
## threaded operation

from threading import Thread


log("Starting the breakfast process")

## create a thread

log("Creating the thread objects")

coffee_thread = threading.Thread(target = some_task, args =  ("making coffee", 5), name = "coffee_thread") 
# toast_thread = threading.Thread(target = some_task, args =  ("making toast", 10), name = "toast_thread") 

log("Created the thread objects")


log("Starting the thread objects!!")

coffee_thread.start()
log("coffee thread started!!")

# toast_thread.start()
# log("toast thread started!!")


## try adding thread.join

for i in range(10): # this is for checking the status of coffee thread whether its active or not
    log(f"***status of coffee thread: , {coffee_thread.is_alive() = }")
    time.sleep(1)


log("finished the breakfast process")




2026-09-02 07:10:19.066784 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.067364 | Creating the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.067841 | Created the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.068031 | Starting the thread objects!! | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.068312 | Starting task: making coffee | thread = coffee_thread | active threads: 10 
2026-09-02 07:10:19.069915 | coffee thread started!! | thread = MainThread | active threads: 10 
2026-09-02 07:10:19.070336 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active threads: 10 
2026-09-02 07:10:20.072551 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active threads: 10 
2026-09-02 07:10:21.074183 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active th

#### what about GIL ?

In [41]:
import sys

print("Python version:", sys.version)

# Available in Python 3.13+.
# True  -> this interpreter has the GIL enabled.
# False -> this is a free-threaded build.
if hasattr(sys, "_is_gil_enabled"):
    print("GIL enabled:", sys._is_gil_enabled())
else:
    print("This Python version does not expose _is_gil_enabled().")

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
GIL enabled: True


#### How are the threads scheduled? 

In [42]:
import threading
import time


def cpu_task(name, seconds):
    """
    Perform CPU-intensive Python work for approximately
    'seconds' seconds.

    There is deliberately no time.sleep() here.
    """

    thread = threading.current_thread()

    print(
        f"{name} STARTED | "
        f"thread={thread.name} | "
        f"native_id={thread.native_id}"
    )
    
    end_time = time.perf_counter() + seconds
    counter = 0
    # Keep doing Python work until the requested duration passes.
    while time.perf_counter() < end_time:
        counter += 1
    print(
        f"{name} FINISHED | "
        f"thread={thread.name} | "
        f"counter={counter:,}"
    )


# ---------------------------------------------------------
# Create two CPU-bound worker threads.
# ---------------------------------------------------------

thread1 = threading.Thread(
    target=cpu_task,
    args=("Task-A", 5),
    name="Worker-A"
)

thread2 = threading.Thread(
    target=cpu_task,
    args=("Task-B", 5),
    name="Worker-B"
)


# ---------------------------------------------------------
# Start both threads.
# ---------------------------------------------------------

start = time.perf_counter()
thread1.start()
thread2.start()


# ---------------------------------------------------------
# Wait for both workers.
# ---------------------------------------------------------

thread1.join()
thread2.join()

elapsed = time.perf_counter() - start

print(f"\nTotal elapsed time: {elapsed:.2f} seconds")

Task-A STARTED | thread=Worker-A | native_id=836116
Task-B STARTED | thread=Worker-B | native_id=836119
Task-A FINISHED | thread=Worker-A | counter=36,011,393
Task-B FINISHED | thread=Worker-B | counter=35,693,464

Total elapsed time: 5.02 seconds


# Experiment 2: Compare sequential and threaded execution

In [ ]:
import threading
import time


def slow_task(name, duration):
    print(f"{name} started")

    time.sleep(duration)

    print(f"{name} finished")


# =========================================================
# SEQUENTIAL VERSION
# =========================================================

print("\nSEQUENTIAL EXECUTION")

start = time.perf_counter()

slow_task("Task-1", 2)
slow_task("Task-2", 2)
slow_task("Task-3", 2)

end = time.perf_counter()

print(f"Sequential time: {end - start:.2f} seconds")


# =========================================================
# MULTITHREADED VERSION
# =========================================================

print("\nMULTITHREADED EXECUTION")

start = time.perf_counter()

threads = []

for i in range(3):

    thread = threading.Thread(
        target=slow_task,
        args=(f"Task-{i + 1}", 2)
    )

    threads.append(thread)

    thread.start()


# Wait until every worker completes
for thread in threads:
    thread.join()


end = time.perf_counter()

print(f"Threaded time: {end - start:.2f} seconds")

# Writing threads in object oriented way

What we have mostly done so far is functional-style threading:

```python
thread = threading.Thread(
    target=some_task,
    args=("coffee", 5)
)

thread.start()
thread.join()
```


The thread executes a function.

In an object-oriented design, the thread's work can instead be encapsulated inside a class:

```python
import threading
import time


class Worker:

    def __init__(self, name):
        self.name = name

    def run(self):
        print(f"{self.name}: starting")

        time.sleep(2)

        print(f"{self.name}: finished")


worker = Worker("Coffee Worker")

thread = threading.Thread(
    target=worker.run
)

thread.start()
thread.join()
```

In [3]:
## functional approach
import threading
import time


def make_coffee():
    print("Making coffee...")
    time.sleep(2)
    print("Coffee ready!")


thread = threading.Thread(
    target=make_coffee
)

thread.start()
thread.join()

In [ ]:
# use an object's method as the target

import threading
import time


class CoffeeMaker:

    def make_coffee(self):
        print("Making coffee...")
        time.sleep(2)
        print("Coffee ready!")


coffee_maker = CoffeeMaker()

thread = threading.Thread(
    target=coffee_maker.make_coffee
)

thread.start()
thread.join()

    Note: when we subclass Thread, we override run() to define what our thread should execute.

In [ ]:
# Subclass threading.Thread

import threading
import time


class CoffeeThread(threading.Thread):

    def run(self):
        print("Making coffee...")
        time.sleep(2)
        print("Coffee ready!")


thread = CoffeeThread()

thread.start()
thread.join()

# Experiment 3 : Race Conditions


The central problem is very simple:

`Multiple threads can access the same piece of shared state, and the final result can depend on the timing and interleaving of their execution.`

That situation is called a race condition.

In [44]:
counter = 0

counter += 1
counter += 1
counter += 1

print(counter)

3


In [47]:
import threading
import time


counter = 0


def increment_counter():
    global counter
    # Read the shared value.
    current_value = counter
    # Deliberately create a window where another thread can execute before we write the result back.
    time.sleep(0.001)
    # Write the updated value.
    counter = current_value + 1


threads = []

# Create 100 threads.
for _ in range(100):

    thread = threading.Thread(
        target=increment_counter
    )

    threads.append(thread)
    thread.start()



# Wait for every thread to finish.
for thread in threads:
    thread.join()


print("Expected:", 100)
print("Actual:  ", counter)

Expected: 100
Actual:   9


    Thread A       Thread B       Thread C
    
    read 0
                   read 0
                                  read 0
    
    write 1
                   write 1
                                  write 1


- three threads worked here but we ended up with counter = 1 instead of counter = 3
- Why is this called a "race"? - Because there is a competition between threads over the timing of operations.

    The correctness of the program depends on the relative timing/interleaving of concurrent operations on shared state.

    
    This is perfectly fine:
    
        Thread A → read database
        Thread B → read database
    They aren't necessarily racing.
    
    But this can race:
    
        Thread A → read inventory = 1
        Thread B → read inventory = 1
        Thread A → decrement inventory
        Thread B → decrement inventory
    
    because both operations depend on shared mutable state.

#### critical section.

A critical section is a portion of code that accesses shared state and must be executed with the required synchronization so that multiple threads don't interfere with one another.

In our example, this is the dangerous sequence:

```
read counter
add 1
write counter
```

We want that entire operation to behave as one logical unit.

Conceptually:

                  Critical Section

              ┌──────────────────────┐
              │ read counter         │
              │ add 1                │
              │ write counter        │
              └──────────────────────┘
We dont want

```
Thread A enters
Thread A reads

Thread B enters
Thread B reads

Thread A writes
Thread B writes
```


We want

```
Thread A
   |
   v
[ read → increment → write ]
   |
   v
Thread B
   |
   v
[ read → increment → write ]
```

#### What is a lock?

A lock is a synchronization mechanism that allows us to say:

"Only one thread at a time may enter this protected section."

In [48]:
import threading
import time


counter = 0

# Create one lock protecting the shared counter.
counter_lock = threading.Lock()


def increment_counter():
    global counter

    # Acquire the lock before touching the shared state.
    with counter_lock:
        current_value= counter
        # We deliberately keep the sleep here to demonstrate that even if this thread pauses, another thread cannot enter this critical section.
        time.sleep(0.001)
        counter = current_value + 1


threads = []
for _ in range(100):
    thread = threading.Thread(target=increment_counter)
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

print("Expected:", 100)
print("Actual:  ", counter)

Expected: 100
Actual:   100


 - The lock therefore protects the atomicity of the logical operation.
 - What is atomicity? --> An operation is atomic if it appears to happen as one indivisible operation from the perspective of other threads. In other words, another thread cannot observe the operation halfway through or interleave its own conflicting operation in the middle.

Consider this example:

```counter += 1```

At the conceptual level, this involves multiple steps:
```
read counter
add 1
write counter
```

Suppose ```counter == 10```.

Without synchronization, two threads could interleave their operations like this:

    Thread A                 Thread B
    
    read 10
                             read 10
    
    calculate 11
                             calculate 11
    
    write 11
                             write 11

The expected result after two increments is 12, but we end up with 11.

The problem is not that the individual read or write necessarily failed. The problem is that the whole read-modify-write operation was not atomic.

We therefore want this:

    Thread A
    ┌─────────────────────┐
    │ read → modify → write│
    └─────────────────────┘
    
    Thread B
                           ┌─────────────────────┐
                           │ read → modify → write│
                           └─────────────────────┘

rather than allowing the operations to interleave.

    A useful way to think about atomicity is: Atomicity is about whether an operation can be observed or interfered with halfway through. A lock is one mechanism for providing that guarantee.
    
    There is another important distinction: atomicity and thread safety are not synonyms. Atomicity is a property of an individual operation or group of operations. Thread safety is the broader property that a component behaves correctly when accessed concurrently. We will see later that achieving thread safety may require more than simply putting a lock around one statement.

### application of threading.lock

In [50]:
class Inventory:
    def __init__(self):
        self.stock = 1

    def purchase(self):
        if self.stock > 0:
            self.stock -= 1
            return True

        return False

    Customer A                     Customer B
    
    check stock > 0
                                   check stock > 0
    
    both see stock = 1
    
    decrement stock
                                   decrement stock

You can potentially sell one item twice. This is the same conceptual problem as our counter.
The critical section is:

        check whether stock exists
                +
        decrement stock

Those operations must be treated as one synchronized operation.

Thread safe implementation

```python
with self.lock:
    if self.stock > 0:
        self.stock -= 1
        return True

    return False

#### some more examples


Identify the shared mutable state and determine which sequence of operations must be performed atomically.

##### Example 1: Shared counter

In [ ]:
## Without synchronization:
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1


        import threading

## with lock
class Counter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()

    def increment(self):
        with self.lock:
            self.value += 1

##### Example 2: Bank account

In [ ]:
class BankAccount:
    def __init__(self):
        self.balance = 100

    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            return True

        return False





    Imagine two threads simultaneously withdraw $80.
    
    They could see:
    
    Initial balance = $100
    
    Thread A: check $100 >= $80 → True
    Thread B: check $100 >= $80 → True
    
    Thread A: balance = $20
    Thread B: balance = -$60

In [56]:
## The synchronized version is:
import threading


class BankAccount:
    def __init__(self):
        self.balance = 100
        self.lock = threading.Lock()

    def withdraw(self, amount):
        with self.lock:
            if self.balance >= amount:
                self.balance -= amount
                return True

            return False

##### Example 3: Inventory

In [ ]:
class Inventory:
    def __init__(self):
        self.stock = 1

    def purchase(self):
        if self.stock > 0:
            self.stock -= 1
            return True

        return False

In [ ]:
import threading


class Inventory:
    def __init__(self):
        self.stock = 1
        self.lock = threading.Lock()

    def purchase(self):
        with self.lock:
            if self.stock > 0:
                self.stock -= 1
                return True

            return False

##### Example 4: Booking a seat

In [ ]:
class Seat:
    def __init__(self):
        self.available = True

    def book(self):
        if self.available:
            self.available = False
            return True

        return False

without syncronization

    Thread A                  Thread B
    
    available?
    → True                    available?
                              → True
    
    book seat                 book seat

In [58]:
# synchronized version
import threading


class Seat:
    def __init__(self):
        self.available = True
        self.lock = threading.Lock()

    def book(self):
        with self.lock:
            if self.available:
                self.available = False
                return True

            return False

##### Example 5: Updating related state

In [ ]:
class ShoppingCart:
    def __init__(self):
        self.total_items = 0
        self.total_value = 0

    def add_item(self, price):
        self.total_items += 1
        self.total_value += price

A reader could potentially observe:

    total_items = 5
    total_value = 400

while another thread is halfway through an update and the state temporarily represents an inconsistent combination.

In [ ]:
import threading


class ShoppingCart:
    def __init__(self):
        self.total_items = 0
        self.total_value = 0
        self.lock = threading.Lock()

    def add_item(self, price):
        with self.lock:
            self.total_items += 1
            self.total_value += price

The important concept here is consistency of shared state. Sometimes the critical section exists not because one individual variable is problematic, but because several variables collectively represent one logical state.

# Experiment 4: acquire() and release()

We have so far used:

```python
with lock:
    # critical section
```
    
Python's with syntax is convenient, but underneath it is essentially managing:

```python
lock.acquire()

try:
    # critical section
finally:
    lock.release()
```

Let's understand these operations directly.

```acquire()``` means:

    Attempt to acquire ownership of the lock.
    If nobody currently owns it, the calling thread obtains it immediately.
    If another thread already owns it, the calling thread waits until the lock becomes available.

```release()``` means:

    Release the lock so that another waiting thread can acquire it.


Basic Pattern:
```python
lock.acquire()

try:
    # Critical section
finally:
    lock.release()
```

For example:

```python
import threading


lock = threading.Lock()
lock.acquire()

try:
    print("Thread entered critical section")

    # Shared state modification happens here.

finally:
    lock.release()

print("Thread left critical section")
```

    Thread A
       |
       | acquire()
       v
    ┌───────────────────┐
    │   LOCKED          │
    │                   │
    │ critical section  │
    │                   │
    └─────────┬─────────┘
              |
              | release()
              v
           UNLOCKED

What happens when thread A and thread B both try to access critical section    
    
    Thread A                    Thread B
        
        acquire()
            |
            v
         LOCKED
            |
         critical section       acquire()
                                    |
                                    v
                                 WAITING
                                    |
                                    |
         release()                  |
            |                       |
            └──────────────────────>|
                                    |
                               acquires lock
                                    |
                                    v
                             critical section

The important difference is that with is the safer and more convenient way to express the pattern because Python guarantees that the lock is released when the block is exited, including when an exception occurs.

These are conceptually equivalent:

```python
lock.acquire()

try:
    # Critical section
    update_shared_state()

finally:
    lock.release()
```

and:

```python
with lock:
    # Critical section
    update_shared_state()
```

The second form is essentially using the lock as a context manager. Under the hood, Python calls acquire() when entering the with block and release() when leaving it.

So why would we ever use acquire() directly?

Because acquire() gives you more control over the locking behavior.

For example, you can ask Python not to wait if the lock is currently unavailable:

```python
acquired = lock.acquire(blocking=False)

if acquired:
    try:
        # We successfully acquired the lock.
        update_shared_state()
    finally:
        lock.release()
else:
    # Someone else currently owns the lock.
    print("Could not acquire lock; doing something else.")
```

Here, ```with lock```: cannot directly express the same "try once and don't wait" behavior.

You can also specify a timeout:

```python
acquired = lock.acquire(timeout=2)

if acquired:
    try:
        update_shared_state()
    finally:
        lock.release()
else:
    print("Lock was not acquired within 2 seconds.")
```

So the practical rule is:

```
with lock:
    ...
```
    
is the normal choice when you simply want to protect a critical section.
```
lock.acquire(...)
lock.release()
```

is useful when you need explicit control over acquisition, such as non-blocking acquisition, timeouts, or more complicated synchronization logic.

One subtle but important point: never casually write this:

```python
lock.acquire()

update_shared_state()

lock.release()
```
    
because if update_shared_state() raises an exception, release() may never execute, leaving the lock permanently acquired. That can cause other threads to wait forever. If you need manual acquire()/release(), use try/finally.

# Experiment 5: Lock contention -  what happens when two threads call acquire()

In [62]:
# Two threads competing for one lock

import threading
import time
from datetime import datetime


lock = threading.Lock()


def log(message):
    """
    Print the current time and thread name so that we can
    observe the ordering of events.
    """

    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    thread = threading.current_thread()

    print(
        f"{timestamp} | "
        f"{thread.name:<12} | "
        f"{message}"
    )


def worker():
    log("About to call lock.acquire()")

    # This thread will wait here if another thread already
    # owns the lock.
    lock.acquire()

    try:
        log("Successfully acquired the lock")

        # Hold the lock for 5 seconds.
        log("Entering critical section")
        time.sleep(5)
        log("Leaving critical section")

    finally:
        # Always release the lock.
        lock.release()

        log("Released the lock")


thread_a = threading.Thread(target=worker,name="Worker-A")
thread_b = threading.Thread(target=worker,name="Worker-B")


log("Starting Worker-A")
thread_a.start()

# Give Worker-A enough time to acquire the lock before starting Worker-B.
time.sleep(0.5)

log("Starting Worker-B")
thread_b.start()

thread_a.join()
thread_b.join()

log("Both workers finished")

08:50:35.059 | MainThread   | Starting Worker-A
08:50:35.060 | Worker-A     | About to call lock.acquire()
08:50:35.060 | Worker-A     | Successfully acquired the lock
08:50:35.060 | Worker-A     | Entering critical section
08:50:35.562 | MainThread   | Starting Worker-B
08:50:35.563 | Worker-B     | About to call lock.acquire()
08:50:40.064 | Worker-A     | Leaving critical section
08:50:40.066 | Worker-A     | Released the lock
08:50:40.066 | Worker-B     | Successfully acquired the lock
08:50:40.066 | Worker-B     | Entering critical section
08:50:45.072 | Worker-B     | Leaving critical section
08:50:45.074 | Worker-B     | Released the lock
08:50:45.075 | MainThread   | Both workers finished


what happened in the above code:


    Worker-A:
        acquired lock
        entered critical section
        ...
        ...
        released lock
    
    Worker-B:
        called acquire()
        WAITED
        WAITED
        WAITED
        lock became available
        acquired lock
        entered critical section


This is **lock contention**. Multiple threads want the same resource, but the resource can only be accessed by one thread at a time.

```acquire()``` can also be non-blocking

In [65]:
import threading
import time


lock = threading.Lock()


def worker_a():
    with lock:
        print("Worker-A acquired the lock")
        time.sleep(5)
        print("Worker-A releasing the lock")


def worker_b():
    time.sleep(1)

    print("Worker-B trying to acquire the lock")

    acquired = lock.acquire(blocking=False)

    if acquired:
        try:
            print("Worker-B acquired the lock")
        finally:
            lock.release()
    else:
        print("Worker-B could not acquire the lock")
        print("Worker-B will do something else")


thread_a = threading.Thread(
    target=worker_a,
    name="Worker-A"
)

thread_b = threading.Thread(
    target=worker_b,
    name="Worker-B"
)

thread_a.start()
thread_b.start()

thread_a.join()
thread_b.join()

Worker-A acquired the lock
Worker-B trying to acquire the lock
Worker-B could not acquire the lock
Worker-B will do something else
Worker-A releasing the lock


Here Worker-B doesn't wait for Worker-A.

Instead:

    Worker-A:
        acquire
        |
        | holds lock for 5 sec
        |
        release
    
    
    Worker-B:
        acquire(blocking=False)
              |
              v
          unavailable
              |
              v
          immediately continue

This is useful when waiting is undesirable and the application has an alternative course of action.

using timeout in acquire ---> ```acquire(timeout=...)```

```acquire(timeout=2)``` means try to acquire the lock, but wait at most two seconds.

In [ ]:
import threading
import time


lock = threading.Lock()


def worker_a():
    with lock:
        print("Worker-A acquired the lock")

        time.sleep(5)

        print("Worker-A releasing the lock")


def worker_b():
    time.sleep(1)

    print("Worker-B trying to acquire the lock")

    acquired = lock.acquire(timeout=2)

    if acquired:
        try:
            print("Worker-B acquired the lock")
        finally:
            lock.release()
    else:
        print("Worker-B timed out waiting for the lock")


thread_a = threading.Thread(target=worker_a)
thread_b = threading.Thread(target=worker_b)

thread_a.start()
thread_b.start()

thread_a.join()
thread_b.join()

# Lesson 6: Check-Then-Act

The check-then-act pattern is one of the most important concurrency patterns to understand because many real-world operations naturally have two steps: first we check whether some condition is true, and then we perform an action based on that observation. The problem is that, when multiple threads are involved, another thread can change the shared state between the check and the action.

Example:
```python
if self.stock > 0:
    self.stock -= 1
```

It looks like one logical operation to us, but it actually contains two distinct operations:

    CHECK                         ACT
    
    Is stock > 0?                 Decrease stock
         │                             │
         └──────────── time ───────────┘

    That gap between the check and the action is where another thread can interfere.


Imagine a cinema with one available seat.

```python
class Seat:
    def __init__(self):
        self.available = True

    def book(self):
        if self.available:       # CHECK
            self.available = False  # ACT
            return True

        return False
```
In a sequential program, this appears perfectly safe. If Customer A books the seat first, available becomes False, and Customer B subsequently sees False. But concurrency changes the situation.


Suppose Customer A and Customer B are represented by two threads:

        Initial state:
        
        available = True
        
        
        Thread A                         Thread B
        
        CHECK
        available == True
                                         CHECK
                                         available == True
        
        ACT
        available = False
                                         ACT
                                         available = False

Both customers observed the seat as available. The problem is that the condition was checked before either thread performed the action. Both threads effectively received permission to book the same resource. This is a race condition caused by a check-then-act sequence.


Correct version:

```python
import threading


class Seat:
    def __init__(self):
        self.available = True
        self.lock = threading.Lock()

    def book(self):
        with self.lock:
            # CHECK and ACT are inside the same critical section.
            if self.available:
                self.available = False
                return True

            return False
```


Now the execution is:

        Thread A                         Thread B
        
        acquire lock
            |
            v
        CHECK
        available == True
            |
            v
        ACT
        available = False
            |
            v
        release lock
                                         acquire lock
                                             |
                                             v
                                         CHECK
                                         available == False
                                             |
                                             v
                                         return False


 The second thread cannot perform its check until the first thread has completed its action.

This gives us the crucial property:
```The state observed during the check cannot be changed by another thread before the corresponding action is performed.```
That is the real reason we put the check and act inside the same critical section.

## Check-then-act isn't always obvious

The pattern can be hidden inside apparently innocent code.

For example:
```python
if key not in cache:
    cache[key] = expensive_calculation()

```

This is also check-then-act.

The check is:

```python
key not in cache
```
    
and the action is:
```python
cache[key] = ...
```

Two threads could do:

    Thread A                         Thread B
    
    check key not in cache
    → True
    
                                     check key not in cache
                                     → True
    
    calculate value
                                     calculate value
    
    store value
                                     store value

Whether this is actually a correctness problem depends on what the application requires. Perhaps calculating the value twice is merely inefficient; perhaps the calculation has side effects and doing it twice is incorrect.

This is an important concurrency lesson:


When you see code like this:

```python
if condition:
    modify_shared_state()
```

you should immediately ask:

```Can another thread modify the state between my check and my action?```

If yes, you potentially have a check-then-act race.

```Does the correctness of the action depend on the condition remaining true?```

If yes, the check and action generally need to be protected together.

For example:

```Inventory:```

    check stock > 0
           ↓
    decrement stock

```Bank account```:

    check balance >= amount
           ↓
    deduct amount

```Seat Booking:```

    check available
           ↓
    mark booked
        
```Parking:```

    check capacity available
           ↓
    increment occupancy

These are all similar concurrency patterns that we come accross very often. 

To summarise:


          Shared State
               |
               v
            CHECK
               |
               |  <-- dangerous window
               |
               v
             ACT

If another thread can modify the relevant state during that window, you have a potential race.


Using a lock correctly turns it into:

                 acquire lock
                  |
                  v
                CHECK
                  |
                  v
                 ACT
                  |
                  v
             release lock

### Booking seats with no concurrency protection

In [9]:

import threading
import time


class Seat:
    def __init__(self):
        self.available = True

    def book(self, customer):
        print(f"\n{customer}: checking seat...")

        # CHECK
        if self.available:

            # Artificial delay to make the race visible.
            time.sleep(1)

            # ACT
            self.available = False

            print(f"\n{customer}: BOOKING SUCCESSFUL")
            return True

        print(f"\n{customer}: booking failed")
        return False


seat = Seat()

ramesh = threading.Thread(
    target=seat.book,
    args=("Ramesh",)
)

suresh = threading.Thread(
    target=seat.book,
    args=("Suresh",)
)

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()



Ramesh: checking seat...
Suresh: checking seat...


Suresh: BOOKING SUCCESSFUL
Ramesh: BOOKING SUCCESSFUL



#### making the race condition even more visible using barrier [ just to demonstrate the interleaving of threads ]


    Ramesh checks → available
                        ↓
                  waits at barrier
    
    Suresh checks → available
                        ↓
                  waits at barrier
    
           both threads continue
    
    Ramesh → book
    Suresh   → book

In [10]:
## making the race condition even more visible using barrier

import threading
import time


class Seat:
    def __init__(self):
        self.available = True


seat = Seat()

# Both customers must reach the barrier
# before either is allowed to continue.
barrier = threading.Barrier(2)


def book_seat(customer):

    print(f"{customer}: checking seat")

    # CHECK
    if seat.available:
        print(f"{customer}: seat is available")

        # Wait until BOTH customers have completed the check.
        barrier.wait()

        print(f"{customer}: proceeding with booking")

        # ACT
        seat.available = False

        print(f"{customer}: BOOKING SUCCESSFUL")

    else:
        print(f"{customer}: seat is already booked")


ramesh = threading.Thread(
    target=book_seat,
    args=("Ramesh",)
)

suresh = threading.Thread(
    target=book_seat,
    args=("Suresh",)
)

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()

Ramesh: checking seatSuresh: checking seat
Suresh: seat is available

Ramesh: seat is available
Ramesh: proceeding with booking
Ramesh: BOOKING SUCCESSFUL
Suresh: proceeding with booking
Suresh: BOOKING SUCCESSFUL


## correct handling using lock

keep in mind that this is wrong!

```python
if self.available:

    with self.lock:
        self.available = False

```


correct way is

```python
with self.lock:

if self.available:
    self.available = False
```


Think of this as one atomic business operation:

        CHECK
         +
        ACT
         =
        ONE TRANSACTION

In [18]:
class Seat:
    def __init__(self):
        self.available = True
        self.lock = threading.Lock()

    def book(self, customer):

        with self.lock:

            # CHECK
            if not self.available:
                print(f"{customer}: booking failed")
                return False

            # ACT
            self.available = False

            print(f"{customer}: BOOKING SUCCESSFUL")
            return True



seat = Seat()

ramesh = threading.Thread(target=seat.book, args=("Ramesh",))

suresh = threading.Thread(target=seat.book,args=("Suresh",))

# ramesh.start()
suresh.start()
ramesh.start()

ramesh.join()
suresh.join()


Suresh: BOOKING SUCCESSFUL
Ramesh: booking failed


## Another example of no so obvious race condition - Cache Population

Suppose we have an expensive function:

```python
def fetch_from_database(user_id):
    print(f"Fetching user {user_id} from database...")
    time.sleep(2)
    return f"User-{user_id}"

```

We want to cache the result.

A naive cache implementation might be:

```python
class UserCache:

    def __init__(self):
        self.cache = {}

    def get_user(self, user_id):

        # CHECK
        if user_id in self.cache:
            print("Cache HIT")
            return self.cache[user_id]

        # ACT
        print("Cache MISS")

        user = fetch_from_database(user_id)

        self.cache[user_id] = user

        return user
```
At first glance, this looks fine.

Suppose Ramesh and Suresh both request` get_user(42)` at almost exactly the same time.

    The execution could be:
    
    Ramesh                         Suresh
    -----                         ---
    
    check cache
    42 not present
    
                                  check cache
                                  42 not present
    
    fetch database               fetch database
    
                                  fetch database
    
    store result                 store result

The cache is still correct. We didn't corrupt the dictionary. 
But we have a concurrency bug of a different kind.

Instead of 1 database request we made 2 database requests

If 1,000 requests arrive simultaneously for a cache-missing key, we could potentially generate 1,000 database requests for the same data.

This is sometimes called a `cache stampede` / `thundering herd problem`. And this is why race conditions are interesting: the program doesn't necessarily crash or produce obviously incorrect data.

It can simply become inefficient or overload another system.

#### Lets try to reproduce it

##### No lock: duplicate database work

        Ramesh                         Suresh
          |                              |
          | cache check                  |
          | → MISS                       |
          |                              |
          |                         cache check
          |                         → MISS
          |                              |
          | DB call                      | DB call
          |                              |
          +---------- 2 seconds ---------+

In [20]:
import threading
import time


def fetch_from_database(user_id):
    print(f"DB: fetching user {user_id}")
    time.sleep(2)
    return f"User-{user_id}"


class UserCache:

    def __init__(self):
        self.cache = {}

    def get_user(self, user_id):

        print(f"{threading.current_thread().name}: checking cache")

        # CHECK
        if user_id in self.cache:
            print(f"{threading.current_thread().name}: CACHE HIT")
            return self.cache[user_id]

        print(f"{threading.current_thread().name}: CACHE MISS")

        # ACT
        user = fetch_from_database(user_id)

        self.cache[user_id] = user

        return user


cache = UserCache()

barrier = threading.Barrier(2)


def get_user(customer):

    print(f"{customer}: starting")

    # Both threads reach the same point
    barrier.wait()

    user = cache.get_user(42)

    print(f"{customer}: received {user}")

ramesh = threading.Thread(target=get_user,args=("Ramesh",),name="rameshThread")
suresh = threading.Thread(target=get_user,args=("Suresh",),name="sureshThread")

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()

Ramesh: starting
Suresh: starting
sureshThread: checking cache
sureshThread: CACHE MISS
DB: fetching user 42
rameshThread: checking cache
rameshThread: CACHE MISS
DB: fetching user 42
Suresh: received User-42
Ramesh: received User-42


Both threads see the cache as empty and both perform the expensive database operation.
The cache itself is not necessarily corrupted. The final value may be perfectly correct.
The problem is that we performed work twice when we wanted it performed once.

##### Correctly protected with a lock

In [22]:
# Now we add a lock around the check + database fetch + cache update
import threading
import time


def fetch_from_database(user_id):
    print(f"DB: fetching user {user_id}")
    time.sleep(2)
    return f"User-{user_id}"


class UserCache:

    def __init__(self):
        self.cache = {}
        self.lock = threading.Lock()

    def get_user(self, user_id):

        with self.lock:

            print(f"{threading.current_thread().name}: checking cache")

            # CHECK
            if user_id in self.cache:
                print(f"{threading.current_thread().name}: CACHE HIT")
                return self.cache[user_id]

            print(f"{threading.current_thread().name}: CACHE MISS")

            # ACT
            user = fetch_from_database(user_id)

            self.cache[user_id] = user

            return user


cache = UserCache()

def get_user(customer):

    print(f"{customer}: starting")
    user = cache.get_user(42)
    print(f"{customer}: received {user}")


ramesh = threading.Thread(target=get_user,args=("Ramesh",),name="rameshThread")

suresh = threading.Thread(target=get_user,args=("Suresh",),name="sureshThread")

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()

Ramesh: starting
rameshThread: checking cache
rameshThread: CACHE MISS
DB: fetching user 42
Suresh: starting
Ramesh: received User-42sureshThread: checking cache
sureshThread: CACHE HIT
Suresh: received User-42



Now the flow is 

    Ramesh                         Suresh
      |                              |
      | acquire lock                 |
      |                              |
      | check → MISS                 |
      |                              |
      | DB call (2 sec)              |
      |                              |
      | cache[42] = User-42          |
      |                              |
      | release lock                 |
      |                              |
      |                         acquire lock
      |                              |
      |                         check → HIT
      |                              |
      |                         release lock

However there is still some problem with the above approach

Suppose:

    Ramesh → user 42 → database takes 2 seconds
    Suresh → user 99
    Priya  → user 150

    Ramesh
       |
       +---- LOCK
              |
              +---- DB call for user 42 (2 sec)
              |
           UNLOCK
    
    Suresh
       |
       +---- waits
           
    Priya
       |
       +---- waits


Even though Suresh and Priya are requesting completely different users, they have to wait.

This is why we need lock granularity 

# Lock Granularity and Lock Contention


We have just established that the cache can be made correct with:

```python
with self.lock:
    if user_id in self.cache:
        return self.cache[user_id]

    user = fetch_from_database(user_id)
    self.cache[user_id] = user
```

The next question is whether this is a good concurrent design.

We will examine three implementations:

| Version | Design          | Lock granularity           | Result                                           |
| ------- | --------------- | -------------------------- | ------------------------------------------------ |
| **1**   | No lock         | No synchronization         | Duplicate database work                          |
| **2**   | One global lock | **Coarse-grained locking** | Correct, but unrelated requests block each other |
| **3**   | Per-key locks   | **Fine-grained locking**   | Correct, while allowing more concurrency         |



```A lock should protect the shared state that requires synchronization, but it should not unnecessarily serialize independent work.```

##### One global lock (Coarse-grained locking) - This is coarse-grained locking because one lock protects the entire cache operation.

This is the flow even though the requests are completely independent. A synchronization mechanism can be correct but poorly designed.

    Ramesh                    Suresh
      |                         |
      | acquire lock            |
      |                         |
      | DB(42)                  |
      | 2 seconds               |
      |                         |
      | release                 |
      |                         |
      |                    acquire lock
      |                         |
      |                    DB(43)
      |                    2 seconds
      |                         |
      |                    release

In [24]:
# Coarse-grained locking - This is coarse-grained locking because one lock protects the entire cache operation.

import threading
import time


def fetch_from_database(user_id):
    print(f"DB: fetching user {user_id}")
    time.sleep(2)
    return f"User-{user_id}"


class UserCache:

    def __init__(self):
        self.cache = {}
        self.lock = threading.Lock()

    def get_user(self, user_id):

        with self.lock:

            print(f"{threading.current_thread().name}: checking cache")

            if user_id in self.cache:
                print(f"{threading.current_thread().name}: CACHE HIT")
                return self.cache[user_id]

            print(f"{threading.current_thread().name}: CACHE MISS")

            # Database call happens while holding the lock.
            user = fetch_from_database(user_id)

            self.cache[user_id] = user

            return user


cache = UserCache()

ramesh = threading.Thread(
    target=lambda: cache.get_user(42),
    name="Ramesh"
)

suresh = threading.Thread(
    target=lambda: cache.get_user(43),
    name="Suresh"
)

start = time.time()

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()

print(f"Total time: {time.time() - start:.2f} seconds")

Ramesh: checking cache
Ramesh: CACHE MISS
DB: fetching user 42
Suresh: checking cache
Suresh: CACHE MISS
DB: fetching user 43
Total time: 4.01 seconds


Why can't we simply move the database call outside the lock?

You can  try this:

```python
def get_user(self, user_id):

    with self.lock:

        if user_id in self.cache:
            return self.cache[user_id]

    # Lock released
    user = fetch_from_database(user_id)

    with self.lock:
        self.cache[user_id] = user

    return user
```

Now different users can be fetched concurrently:

    Ramesh ── check ── DB(42) ── update
    Suresh ── check ── DB(43) ── update

That looks much better.

But we have introduced our original race condition again.

Consider two requests for the same user:

    Ramesh → get_user(42)
    Suresh → get_user(42)

Execution:
    
    Ramesh                    Suresh
      |                         |
      | check → MISS            |
      |                         |
      |                    check → MISS
      |                         |
      | DB(42)                  | DB(42)
      |                         |

We are back to duplicate database work. Therefore, simply making the lock smaller is not enough. We have to think carefully about what state needs synchronization.

##### Per-key locking (Fine-grained locking)

A simple teaching implementation is to have a separate lock for each key.


Now consider:

    Ramesh → get_user(42)
    Suresh → get_user(42)
    Priya  → get_user(43)

The synchronization looks conceptually like:

                 UserCache
                    |
          ┌─────────┴─────────┐
          ↓                   ↓
       lock[42]             lock[43]
          |                   |
       Ramesh              Priya
       Suresh


Therefore:

    Ramesh ── DB(42) ────────────┐
                                 │
    Suresh ── waits ─────────────┘
    
    Priya ── DB(43) ──────────────→ independently



A coarse lock protects a large unit of shared state: entire cache

A fine-grained lock protects a smaller unit: individual cache key 


But, obviously there are some trade offs:


With more locks, we introduce additional complexity:

Coarse-grained: 1 lock --> simple --> less concurrency

Fine-grained: many locks --> more concurrency --> more complicated

Fine-grained locking can also introduce problems such as:

    deadlocks when multiple locks are acquired in different orders;
    increased memory/management overhead;
    more complicated reasoning about correctness;
    difficulty cleaning up unused per-key locks.

In [26]:
import threading
import time


def fetch_from_database(user_id):
    print(f"DB: fetching user {user_id}")
    time.sleep(2)
    return f"User-{user_id}"


class UserCache:

    def __init__(self):
        self.cache = {}

        # One lock for managing the lock dictionary itself.
        self.lock = threading.Lock()

        # Separate lock for each user_id.
        self.key_locks = {}

    def get_lock_for_key(self, user_id):

        with self.lock:
            if user_id not in self.key_locks:
                self.key_locks[user_id] = threading.Lock()

            return self.key_locks[user_id]

    def get_user(self, user_id):

        key_lock = self.get_lock_for_key(user_id)

        # Only requests for the SAME key block each other.
        with key_lock:

            if user_id in self.cache:
                print(
                    f"{threading.current_thread().name}: CACHE HIT"
                )
                return self.cache[user_id]

            print(
                f"{threading.current_thread().name}: CACHE MISS"
            )

            user = fetch_from_database(user_id)

            self.cache[user_id] = user

            return user

`Note`: 
Thhe per-key implementation above is deliberately designed for learning. Production cache implementations often use more sophisticated techniques such as single-flight/request coalescing, futures/promises, striped locks, or cache libraries with their own concurrency guarantees. We do not need those yet.

We are just trying to understand why lock placement and lock scope are design decisions. 


# Deadlocks	

**Deadlocks in Multithreading**

A deadlock occurs when two or more threads are permanently blocked because each thread is waiting for a resource held by another thread.

We have already learned that a lock makes a critical section safe. Deadlocks are the problem that can arise when we start using multiple locks.

The simplest example is a bank transfer.

Suppose we have two accounts Account A Account B

and transferring money from A to B requires locking both accounts.


Imagine:

    Ramesh: transfer A → B
    Suresh: transfer B → A

Suppose the implementation acquires the locks in the order in which the accounts appear:

    lock(source)
    lock(destination)

The execution can become:

    Ramesh                         Suresh
    
    acquire Account A
    
                                  acquire Account B
    
    waiting for Account B
    
                                  waiting for Account A

Now neither thread can proceed:

    Ramesh → waiting for Suresh
    Suresh → waiting for Ramesh

This is a deadlock. Neither thread will release its first lock because it is waiting for the second lock.

**What actually causes a deadlock?**

There are four classic conditions, commonly called the Coffman conditions. A deadlock requires all four to be present:

| Condition            | Meaning                                                           |
| -------------------- | ----------------------------------------------------------------- |
| **Mutual exclusion** | A resource can be held by only one thread at a time               |
| **Hold and wait**    | A thread holds one resource while waiting for another             |
| **No preemption**    | A resource cannot simply be taken away from the thread holding it |
| **Circular wait**    | Thread A waits for B, B waits for A (or a longer cycle)           |


Our example satisfies all four:

    Ramesh holds A → waits for B
    Suresh holds B → waits for A

The most useful one to recognize in everyday code is circular wait.

**Deadlock vs race condition**

These are easy to confuse, so keep the distinction clear.

`Race condition --> Threads proceed, but the result depends on timing --> incorrect/unpredictable behavior`

    Ramesh ──┐
             ├── both modify shared state
    Suresh ──┘
                  ↓
              wrong result

`Deadlock --> Threads cannot proceed at all --> permanent waiting`

    Ramesh → waiting for Suresh
    Suresh → waiting for Ramesh
                  ↓
              nobody moves


In [27]:
import threading
import time


lock_a = threading.Lock()
lock_b = threading.Lock()

def ramesh_task():
    print("Ramesh: acquiring lock A")
    lock_a.acquire()
    print("Ramesh: acquired lock A")
    # Give Suresh a chance to acquire lock B.
    time.sleep(1)
    print("Ramesh: trying to acquire lock B")
    lock_b.acquire()
    print("Ramesh: acquired lock B")
    lock_b.release()
    lock_a.release()


def suresh_task():

    print("Suresh: acquiring lock B")
    lock_b.acquire()
    print("Suresh: acquired lock B")
    # Give Ramesh a chance to acquire lock A.
    time.sleep(1)
    print("Suresh: trying to acquire lock A")
    lock_a.acquire()
    print("Suresh: acquired lock A")
    lock_a.release()
    lock_b.release()

ramesh = threading.Thread(target=ramesh_task,name="RameshThread")
suresh = threading.Thread(target=suresh_task,name="SureshThread")

ramesh.start()
suresh.start()

ramesh.join()
suresh.join()

print("Program finished")

Ramesh: acquiring lock A
Ramesh: acquired lock A
Suresh: acquiring lock B
Suresh: acquired lock B
Ramesh: trying to acquire lock BSuresh: trying to acquire lock A



KeyboardInterrupt: 

**How do we fix it?**

The simplest solution is consistent lock ordering.

The problem was:
    Ramesh: A → B
    Suresh: B → A

Instead, establish a rule that `Always acquire Account A before Account B.`

Then both operations follow: `A → B`

Even if the transfers are in opposite directions.

For example:

```python
def transfer(account1, account2):

    # Always acquire locks in a consistent order.
    first, second = sorted(
        [account1, account2],
        key=lambda account: account.id
    )

    with first.lock:
        with second.lock:
            # Perform transfer
            ...
```
Now the possible execution is:



        Ramesh                         Suresh
        
        acquire A
        
                                      tries A
                                      ↓
                                      waits
        
        acquire B
        
        perform transfer
        
        release B
        release A
        
                                      acquire A
                                      acquire B
                                      perform transfer

There is waiting, but there is no circular waiting. This is one of the most practical deadlock-prevention techniques
`If multiple locks must be acquired, always acquire them in the same global order.`

In [29]:
import threading
import time


class BankAccount:

    def __init__(self, account_id, owner, balance):
        self.account_id = account_id
        self.owner = owner
        self.balance = balance
        self.lock = threading.Lock()


def transfer(source, destination, amount):

    print(
        f"{threading.current_thread().name}: "
        f"transferring ₹{amount} from "
        f"{source.owner} to {destination.owner}"
    )

    
# IMPORTANT -> Always acquire locks in the same global order.
    
    ## We are using account_id to arrive at globally agreed ordering rule.
    if source.account_id < destination.account_id:
        first = source
        second = destination
    else:
        first = destination
        second = source


    # first, second = sorted( [source, destination],    key=lambda account: account.account_id)

    # First acquire the lower-ID account's lock.
    with first.lock:

        print(
            f"{threading.current_thread().name}: "
            f"acquired lock for {first.owner}"
        )

        # Simulate some work so that we can observe
        # the threads interacting.
        time.sleep(1)

        # Then acquire the higher-ID account's lock.
        with second.lock:

            print(
                f"{threading.current_thread().name}: "
                f"acquired lock for {second.owner}"
            )

            # Both accounts are now protected.
            if source.balance < amount:
                print(
                    f"{threading.current_thread().name}: "
                    f"insufficient balance"
                )
                return

            source.balance -= amount
            destination.balance += amount

            print(
                f"{threading.current_thread().name}: "
                f"transfer completed"
            )



# Create two accounts

ramesh_account = BankAccount(
    account_id=1,
    owner="Ramesh",
    balance=1000
)

suresh_account = BankAccount(
    account_id=2,
    owner="Suresh",
    balance=1000
)



# Two transfers in opposite directions
ramesh_to_suresh = threading.Thread(target=transfer,args=(ramesh_account, suresh_account, 100),name="RameshThread")
suresh_to_ramesh = threading.Thread(target=transfer,args=(suresh_account, ramesh_account, 200),name="SureshThread")

# Start both threads
ramesh_to_suresh.start()
suresh_to_ramesh.start()


# Wait for both transfers to finish
ramesh_to_suresh.join()
suresh_to_ramesh.join()


# Final balances
print()
print(f"Ramesh balance: ₹{ramesh_account.balance}")
print(f"Suresh balance: ₹{suresh_account.balance}")

RameshThread: transferring ₹100 from Ramesh to SureshSureshThread: transferring ₹200 from Suresh to Ramesh
SureshThread: acquired lock for Ramesh

SureshThread: acquired lock for Suresh
SureshThread: transfer completed
RameshThread: acquired lock for Ramesh
RameshThread: acquired lock for Suresh
RameshThread: transfer completed

Ramesh balance: ₹1100
Suresh balance: ₹900


**Practical rules for avoiding deadlocks**

For normal Python application development, you do not need to memorize complicated deadlock theory. These rules cover most situations:

1. *Prefer a single lock when possible* -If one lock can safely protect the state, it is usually simpler than coordinating multiple locks.

2. *Acquire multiple locks in a consistent order.*
lock A → lock B → lock C
Every thread follows the same order.

3. *Keep critical sections small.*

Don't hold locks while doing unnecessarily slow operations.

4. *Avoid calling unknown/external code while holding a lock.*

For example, calling another object's method while holding a lock can be dangerous if that method tries to acquire another lock.

5. *Use timeouts when appropriate.*

Instead of:

```lock.acquire()```

you can sometimes use:

```python
if lock.acquire(timeout=2):
    try:
        ...
    finally:
        lock.release()
else:
    print("Could not acquire lock")
```

A timeout doesn't automatically prevent deadlocks, but it can prevent a thread from waiting forever.


# Queue and Producer–Consumer	


So far, we have primarily dealt with shared state:

    Thread A ─────┐
                  ↓
             Shared object
                  ↑
    Thread B ─────┘

This creates problems because multiple threads can read and modify the same state. Locks can make this safe, but synchronization becomes increasingly complicated as the application grows.

A different approach is to let threads communicate through a thread-safe queue.


    Producer threads
          |
          | put()
          ↓
    +-------------+
    | queue.Queue |
    +-------------+
          |
          | get()
          ↓
    Consumer threads

The queue takes care of the synchronization internally. Multiple threads can safely call put() and get().

Imagine an order-processing system.

Ramesh, Suresh, and Priya generate orders, and worker threads process them.

The execution looks roughly like:

                Queue
                  |
        ┌─────────┴─────────┐
        ↓                   ↓
    Worker-1             Worker-2
        |                   |
    Order-101           Order-102
        |                   |
    Order-103           Order-104

The important point is that the workers don't need to coordinate with each other over which order they should process.

The queue handles that.


            PRODUCERS
           /    |    \
          ↓     ↓     ↓
       +----------------+
       |     Queue      |
       +----------------+
          ↓     ↓     ↓
       WORKERS / CONSUMERS

In [30]:
import threading
import queue
import time

orders = queue.Queue()

def process_order(order):
    print(f"{threading.current_thread().name}: processing {order}")
    time.sleep(1)
    print(f"{threading.current_thread().name}: completed {order}")


def worker():
    while True:

        order = orders.get()

        if order is None:
            break

        process_order(order)

        # Tell the queue that this item is finished.
        orders.task_done()


# Create workers
worker1 = threading.Thread(target=worker,name="Worker-1")
worker2 = threading.Thread(target=worker,name="Worker-2")

worker1.start()
worker2.start()


# Producers add orders
orders.put("Order-101")
orders.put("Order-102")
orders.put("Order-103")
orders.put("Order-104")


# Wait until all orders have been processed.
orders.join()


# Tell workers to stop.
orders.put(None)
orders.put(None)

worker1.join()
worker2.join()

print("All orders processed")

Worker-1: processing Order-101Worker-2: processing Order-102

Worker-1: completed Order-101Worker-2: completed Order-102
Worker-2: processing Order-103

Worker-1: processing Order-104
Worker-2: completed Order-103Worker-1: completed Order-104

All orders processed




**Why is Queue thread-safe?**

Suppose two workers execute ```order = orders.get()``` at approximately the same time.

The queue internally handles the synchronization necessary to ensure that an item isn't accidentally handed to both workers.

| Method        | Meaning                                                         |
| ------------- | --------------------------------------------------------------- |
| `put(item)`   | Add work to the queue                                           |
| `get()`       | Retrieve/remove one item                                        |
| `task_done()` | Tell the queue that the retrieved item's processing is finished |
| `join()`      | Wait until all queued work has received `task_done()`           |


`thread.join()` waits for a particular thread to terminate, whereas `queue.join()` waits for all work submitted to the queue to be marked complete.

# ThreadPoolExecutor and Future	


Until now, we have created threads manually:
    
    thread1 = threading.Thread(...)
    thread2 = threading.Thread(...)
    thread3 = threading.Thread(...)

This works well when we have a small number of known tasks, but it becomes inconvenient when an application has hundreds or thousands of tasks. We do not want to create a new thread for every task, manage their lifecycle individually, and decide when each thread should terminate. Creating too many threads can also increase memory usage and scheduling overhead.

A thread pool solves this by creating a fixed number of worker threads and allowing us to submit many tasks to those workers. The pool manages which worker executes which task and reuses the workers for subsequent tasks.

Conceptually, instead of doing this:

    Task 1 → Thread 1
    Task 2 → Thread 2
    Task 3 → Thread 3
    Task 4 → Thread 4
    Task 5 → Thread 5

we have:

                 Thread Pool
                      |
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       Worker 1    Worker 2    Worker 3
          ↑           ↑           ↑
          └───────────┼───────────┘
                      |
                    Tasks


If there are 100 tasks and the pool has only 3 workers, the three workers execute tasks concurrently and pick up additional tasks as they become available. We therefore control the maximum number of concurrent worker threads without manually managing 100 threads.

Python provides this abstraction through `concurrent.futures.ThreadPoolExecutor`.

For reference, here is how we manually create threads

```python
threads = []

for order in orders:

    thread = threading.Thread(
        target=process_order,
        args=(order,)
    )

    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

```


We have to manage the threads ourselves.

With a thread pool:

```python
with ThreadPoolExecutor(max_workers=5) as executor:

    futures = [
        executor.submit(process_order, order)
        for order in orders
    ]

    results = [
        future.result()
        for future in futures
    ]

```


The executor manages the worker threads for us.

`threading.Thread` gives you low-level control over individual threads. `ThreadPoolExecutor` gives you a higher-level abstraction for executing many independent tasks concurrently. For most applications where you simply have many independent I/O-bound tasks, the executor is usually the more convenient abstraction.

##### Basic ThreadPoolExecutor

Consider a simple function that processes an order. The time.sleep() represents some I/O operation such as calling a database, an HTTP service, or reading from a remote system.

```python
from concurrent.futures import ThreadPoolExecutor
import time


def process_order(order_id):
    print(f"Processing {order_id}")

    # Simulate an I/O operation.
    time.sleep(2)

    print(f"Completed {order_id}")

    return f"{order_id} processed"


with ThreadPoolExecutor(max_workers=3) as executor:

    future1 = executor.submit(process_order, "Order-101")
    future2 = executor.submit(process_order, "Order-102")
    future3 = executor.submit(process_order, "Order-103")

    result1 = future1.result()
    result2 = future2.result()
    result3 = future3.result()

print(result1)
print(result2)
print(result3)

```


`executor.submit()` does not execute the function in the current thread. It submits the task to the pool and returns a Future object immediately.

With three workers, the three orders can execute concurrently:

    Worker 1 → Order-101 ── 2 sec ── complete
    Worker 2 → Order-102 ── 2 sec ── complete
    Worker 3 → Order-103 ── 2 sec ── complete

The total execution time is therefore approximately two seconds rather than six seconds, assuming the simulated work is I/O-bound and the system can overlap it.

In [33]:
from concurrent.futures import ThreadPoolExecutor
import time


def process_order(order_id):
    print(f"Processing {order_id}")

    # Simulate an I/O operation.
    time.sleep(2)

    print(f"Completed {order_id}")

    return f"{order_id} processed"


with ThreadPoolExecutor(max_workers=3) as executor:

    future1 = executor.submit(process_order, "Order-101")
    future2 = executor.submit(process_order, "Order-102")
    future3 = executor.submit(process_order, "Order-103")

    result1 = future1.result()
    result2 = future2.result()
    result3 = future3.result()

print(result1)
print(result2)
print(result3)

Processing Order-101Processing Order-102

Processing Order-103
Completed Order-102
Completed Order-101
Completed Order-103
Order-101 processed
Order-102 processed
Order-103 processed


What exactly is a Future?

A Future is an object representing the eventual result of a computation.

When we write 
```python 
future = executor.submit(process_order, "Order-101")
```

the function may still be running when submit() returns. The Future gives us a handle through which we can later interact with that computation.

The most important method is `future.result()`

It returns the function's result when the task has completed. If the task is still running, result() waits for it to finish.

Therefore, `result = future.result()`

means approximately `Give me the result of this background task. If it has not finished yet, wait until it does`

This is similar to the `join()` concept we learned earlier, but there is an important difference: `join()` waits for a thread, whereas `Future.result()` waits for a task and gives you its return value.

**The thread pool reuses workers**

Suppose we have:

```python
with ThreadPoolExecutor(max_workers=2) as executor:

    futures = [
        executor.submit(process_order, order)
        for order in range(1, 7)
    ]
```
    
There are six tasks but only two worker threads. The execution conceptually looks like this:

    Worker 1 → Task 1 → Task 3 → Task 5
    Worker 2 → Task 2 → Task 4 → Task 6

The executor does not create six threads.

Suppose Ramesh, Suresh, and Priya each have orders that need to be processed. We can submit all of them to a pool.

```python
from concurrent.futures import ThreadPoolExecutor
import threading
import time


def process_order(order_id):

    thread_name = threading.current_thread().name

    print(
        f"{thread_name}: processing {order_id}"
    )

    # Simulate I/O.
    time.sleep(2)

    print(
        f"{thread_name}: completed {order_id}"
    )

    return f"{order_id} completed"


orders = [
    "Order-Ramesh",
    "Order-Suresh",
    "Order-Priya",
    "Order-Anil",
    "Order-Kavita",
]


with ThreadPoolExecutor(max_workers=3) as executor:

    futures = []

    for order in orders:

        future = executor.submit(
            process_order,
            order
        )

        futures.append(future)

    for future in futures:

        result = future.result()

        print(f"Result: {result}")

```
There are five tasks but only three workers. Initially, three tasks can execute:

    Worker 1 → Order-Ramesh
    Worker 2 → Order-Suresh
    Worker 3 → Order-Priya

When one finishes, that worker becomes available and picks up another task:

    Worker 1 → Order-Anil
    Worker 2 → Order-Kavita

The pool therefore provides controlled concurrency.
    

**What happens if a task raises an exception?**

This is another important advantage of Future. Suppose:

```python
def process_order(order_id):

    if order_id == "Order-102":
        raise ValueError("Invalid order")

    return f"{order_id} processed"
```

The worker thread encounters the exception, but the exception can be retrieved through the Future.

```python
from concurrent.futures import ThreadPoolExecutor


def process_order(order_id):

    if order_id == "Order-102":
        raise ValueError("Invalid order")

    return f"{order_id} processed"


with ThreadPoolExecutor(max_workers=3) as executor:

    future = executor.submit(
        process_order,
        "Order-102"
    )

    try:
        result = future.result()
        print(result)

    except ValueError as error:
        print(f"Task failed: {error}")
```

So the exception is not silently lost. Calling future.result() raises the exception that occurred in the worker. This is an important distinction from simply starting a Thread and forgetting about it.

`submit()` versus `map()`

There is another convenient API: `executor.map()`

For example:

```python
from concurrent.futures import ThreadPoolExecutor
import time


def process_order(order_id):

    time.sleep(2)

    return f"{order_id} completed"


orders = [
    "Order-101",
    "Order-102",
    "Order-103",
    "Order-104",
]


with ThreadPoolExecutor(max_workers=3) as executor:

    results = executor.map(
        process_order,
        orders
    )

    for result in results:
        print(result)
```

`map()` is convenient when we simply want to apply the same function to many inputs.

`submit()` is more flexible because each submission gives us an individual Future, allowing us to inspect, wait for, or handle each task independently.

Consider:

```
future1 = executor.submit(task1)
future2 = executor.submit(task2)

result1 = future1.result()
result2 = future2.result()
```

Calling `future1.result()` does not mean `task2` stops running.Both tasks have already been submitted to the pool and can execute concurrently.
The main thread is simply waiting for `future1` to finish.

    Main thread
        |
        +── submit task1 ──→ Worker 1
        |
        +── submit task2 ──→ Worker 2
        |
        +── wait for future1
        |
        +── wait for future2

The waiting happens in the main thread, not necessarily in the workers.

**What does the with block do?**

We have been writing:

```python
with ThreadPoolExecutor(max_workers=3) as executor:
    ...

```
    
This is the recommended way to use the executor.

When the block exits, the executor shuts down and waits for the submitted tasks to finish.

Conceptually:

        enter with
            ↓
        create thread pool
            ↓
        submit tasks
            ↓
        tasks execute
            ↓
        leave with
            ↓
        shutdown executor
            ↓
        wait for workers/tasks

This means we generally don't need to manually create, start, join, and clean up every worker thread.

# Event, Semaphore, Condition, RLock	


# Exceptions/results/cancellation	


# I/O-bound vs CPU-bound demonstration	


# Threading vs multiprocessing vs asyncio	


# One final practical example	